# Myanmar Headline Generation with LSTM + Attention
## Improved Version with Fixes

**Key Improvements:**
1. ✅ Fixed preprocessing consistency (same regex for text and headlines)
2. ✅ Added teacher forcing during training
3. ✅ Fixed attention mechanism
4. ✅ Added validation loop
5. ✅ Added beam search for better inference
6. ✅ Added BLEU score evaluation
7. ✅ Better hyperparameters and training

In [1]:
!pip install gensim sacrebleu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 13.5 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from gensim.models import KeyedVectors
from tqdm import tqdm
import re
import pickle
from pathlib import Path
from collections import Counter
from sacrebleu.metrics import BLEU

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Text Preprocessor (FIXED)

In [17]:
class MyanmarTextPreprocessor():
    """Fixed preprocessor with consistent regex pattern"""
    def __init__(self, dict_path: str, stop_path: str):
        self.dictionary = self.load_dictionary(dict_path)
        self.stopwords = self.load_stopwords(stop_path)
        # FIXED: Use consistent regex pattern
        self.syllable_pattern = r"(([A-Za-z0-9]+)|[က-အ|ဥ|ဦ](င်္|[က-အ][ှ]*[့း]*[်]|္[က-အ]|[ါ-ှႏꩻ][ꩻ]*){0,}|.)"

    def load_dictionary(self, dict_path):
        dictionary = set()
        with open(dict_path, 'r', encoding='utf-8') as f:
            for line in f:
                word = line.strip()
                if word:
                    dictionary.add(word)
        return dictionary

    def load_stopwords(self, stopword_path):
        stopwords = set()
        with open(stopword_path, 'r', encoding='utf-8') as f:
            for line in f:
                word = line.strip()
                if word:
                    stopwords.add(word)
        return stopwords

    def merge_with_dictionary(self, syllables):
        """Merge syllables into words using dictionary"""
        merged_tokens = []
        i = 0
        while i < len(syllables):
            matched = False
            # Try to match longest possible word first
            for j in range(len(syllables), i, -1):
                combined = ''.join(syllables[i:j])
                if combined in self.dictionary:
                    merged_tokens.append(combined)
                    i = j
                    matched = True
                    break
            if not matched:
                merged_tokens.append(syllables[i])
                i += 1
        return merged_tokens

    def tokenize(self, text: str, use_dict_merge: bool = True, remove_stopwords: bool = True):
        """FIXED: Consistent tokenization for both text and headlines"""
        # Apply syllable segmentation
        text = re.sub(self.syllable_pattern, r"\1 ", text)
        syllables = text.strip().split()

        # Optionally merge with dictionary
        if use_dict_merge:
            tokens = self.merge_with_dictionary(syllables)
        else:
            tokens = syllables

        # Optionally remove stopwords
        if remove_stopwords:
            tokens = [token for token in tokens if token not in self.stopwords]

        return tokens

    def preprocessing(self, text: str):
        """Legacy method for compatibility"""
        return ' '.join(self.tokenize(text, use_dict_merge=True, remove_stopwords=True))

## 2. Load and Preprocess Data

In [18]:
# Paths
DICT_PATH = "/content/drive/MyDrive/NLP Project/dict-words.txt"
STOPWORDS_PATH = "/content/drive/MyDrive/NLP Project/stopwords.txt"
DATA_PATH = "/content/drive/MyDrive/NLP Project/Headline Generator Dataset/headline_corpus.csv"
FASTTEXT_PATH = "/content/drive/MyDrive/NLP Project/Headline Generator Dataset/cc.my.300.vec"
CACHE_FILE = "preprocessed_v2.pkl"

# Load data
df = pd.read_csv(DATA_PATH)
texts = df["text"].astype(str).tolist()
headlines = df["headline"].astype(str).tolist()

print(f"Loaded {len(texts)} articles")

Loaded 16934 articles


In [68]:
# ============================================
# COPY THIS ENTIRE CELL - REPLACE YOUR PREPROCESSING SECTION
# ============================================

# Initialize preprocessor
processor = MyanmarTextPreprocessor(DICT_PATH, STOPWORDS_PATH)

# ⭐ IMPORTANT: Use new cache filename to force reprocessing
CACHE_FILE = "preprocessed_FIXED_consistent.pkl"

# Delete old caches to ensure clean start
import os
for old_cache in ["preprocessed_optimized.pkl", "preprocessed_syllable_fixed.pkl", "preprocessed.pkl", "preprocessed_v2.pkl"]:
    if os.path.exists(old_cache):
        os.remove(old_cache)
        print(f"✓ Deleted old cache: {old_cache}")

if Path(CACHE_FILE).exists():
    print("\nLoading cached preprocessed data...")
    with open(CACHE_FILE, "rb") as f:
        data = pickle.load(f)
        tokenized_texts = data["tokenized_texts"]
        tokenized_headlines = data["tokenized_headlines"]
else:
    print("\n" + "="*60)
    print("PREPROCESSING WITH CONSISTENT SETTINGS")
    print("="*60)
    print("Using: use_dict_merge=False, remove_stopwords=False for BOTH\n")

    tokenized_texts = []
    tokenized_headlines = []

    for text, headline in tqdm(zip(texts, headlines), total=len(texts), desc="Processing"):
        # ⭐ CRITICAL: SAME SETTINGS FOR BOTH!
        tokenized_texts.append(
            processor.tokenize(text, use_dict_merge=False, remove_stopwords=False)
        )
        tokenized_headlines.append(
            processor.tokenize(headline, use_dict_merge=False, remove_stopwords=False)
        )

    # Save cache
    with open(CACHE_FILE, "wb") as f:
        pickle.dump({
            "tokenized_texts": tokenized_texts,
            "tokenized_headlines": tokenized_headlines
        }, f)
    print("\n✓ Saved preprocessed cache")

# ============================================
# VERIFICATION - CHECK IF IT WORKED
# ============================================

print("\n" + "="*60)
print("VERIFICATION")
print("="*60)

print(f"\nTotal examples: {len(tokenized_texts):,}")

# Show samples
print(f"\nSample text tokens (first 25):")
print(f"  {tokenized_texts[0][:25]}")

print(f"\nSample headline tokens (first 15):")
print(f"  {tokenized_headlines[0][:15]}")

# Check vocabulary overlap
print("\nChecking vocabulary overlap (using first 1000 examples)...")

text_vocab = set()
headline_vocab = set()

for t in tokenized_texts[:1000]:
    text_vocab.update(t)

for h in tokenized_headlines[:1000]:
    headline_vocab.update(h)

overlap = text_vocab & headline_vocab
overlap_pct = len(overlap) / len(headline_vocab) * 100

print(f"\n  Text vocabulary:     {len(text_vocab):,} unique tokens")
print(f"  Headline vocabulary: {len(headline_vocab):,} unique tokens")
print(f"  Overlap:             {len(overlap):,} tokens ({overlap_pct:.1f}%)")

# ============================================
# DIAGNOSTIC
# ============================================

if overlap_pct < 70:
    print("\n" + "❌"*30)
    print("ERROR: Overlap still too low!")
    print("❌"*30)

    # Show what's different
    headline_only = headline_vocab - text_vocab
    print(f"\nTokens in headlines but NOT in texts: {len(headline_only)}")
    print(f"Examples: {list(headline_only)[:30]}")

    # Check if dictionary merge happened
    merged_words = [w for w in headline_only if len(w) > 3]
    if merged_words:
        print(f"\n⚠️  Found {len(merged_words)} merged words in headlines!")
        print(f"Examples: {merged_words[:20]}")
        print("\n→ This means use_dict_merge=True is STILL being used somewhere!")

    raise ValueError("Preprocessing is still inconsistent. Check code above.")

elif overlap_pct < 85:
    print("\n⚠️  Overlap is acceptable but could be better.")
    print("   This might be okay if headlines use some unique punctuation/style.")

else:
    print("\n" + "✅"*30)
    print("SUCCESS! Vocabulary overlap is excellent!")
    print("✅"*30)
    print("\nYou can now proceed to vocabulary building and training.")
    print("Expected results:")
    print("  - Vocab size: ~20-25k words (down from 68k)")
    print("  - Epoch 5 loss: ~3.0-3.5 (was 5.5+)")
    print("  - Epoch 15 loss: ~2.0-2.5 (was 5.7)")

print("\n" + "="*60)


Loading cached preprocessed data...

VERIFICATION

Total examples: 16,934

Sample text tokens (first 25):
  ['မော်လ်', 'တာ', 'ကမ်း', 'လွန်', 'မှာ', 'တိမ်း', 'မှောက်', 'သွား', 'တဲ့', 'လှေ', 'ကို', 'ဖေ', 'ဖော်', 'ဝါ', 'ရီ', '၂', '၃', 'ရက်', 'သော', 'ကြာ', 'နေ့', 'က', 'ကယ်', 'ဆယ်', 'ခဲ့']

Sample headline tokens (first 15):
  ['မော်လ်', 'တာ', 'ကမ်း', 'လွန်', '၌', 'လှေ', 'မှောက်', 'မှု', 'ရွှေ့', 'ပြောင်း', 'နေ', 'ထိုင်', 'သူ', '၅', 'ဦး']

Checking vocabulary overlap (using first 1000 examples)...

  Text vocabulary:     3,874 unique tokens
  Headline vocabulary: 1,192 unique tokens
  Overlap:             1,184 tokens (99.3%)

✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅
SUCCESS! Vocabulary overlap is excellent!
✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅

You can now proceed to vocabulary building and training.
Expected results:
  - Vocab size: ~20-25k words (down from 68k)
  - Epoch 5 loss: ~3.0-3.5 (was 5.5+)
  - Epoch 15 loss: ~2.0-2.5 (was 5.7)



## 3. Build Vocabulary and Embeddings

In [69]:
# Load FastText embeddings
print("Loading FastText vectors...")
ft = KeyedVectors.load_word2vec_format(FASTTEXT_PATH)
EMBEDDING_DIM = 300

# Build vocabulary
counter = Counter()
for t in tokenized_texts + tokenized_headlines:
    counter.update(t)

# Keep words appearing at least twice
vocab = ["<pad>", "<unk>", "<sos>", "<eos>"] + [w for w, c in counter.items() if c >= 2]

word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
vocab_size = len(vocab)

print(f"Vocab size: {vocab_size}")
print(f"Coverage in FastText: {sum(1 for w in vocab if w in ft) / len(vocab) * 100:.2f}%")

Loading FastText vectors...
Vocab size: 9084
Coverage in FastText: 66.62%


In [22]:
# Create embedding matrix
embedding_matrix = np.random.normal(scale=0.6, size=(vocab_size, EMBEDDING_DIM))

# Replace with FastText vectors if available
for word, idx in word2idx.items():
    if word in ft:
        embedding_matrix[idx] = ft[word]

# Zero out padding
embedding_matrix[word2idx["<pad>"]] = 0

# Convert to PyTorch tensor
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embedding_matrix = torch.tensor(embedding_matrix, dtype=torch.float).to(DEVICE)

print(f"Embedding matrix shape: {embedding_matrix.shape}")
print(f"Using device: {DEVICE}")

Embedding matrix shape: torch.Size([9084, 300])
Using device: cuda


In [23]:
# Run this to see actual headline lengths
headline_lengths = [len(h) for h in tokenized_headlines]
print(f"Mean: {np.mean(headline_lengths):.1f}")
print(f"Median: {np.median(headline_lengths):.1f}")
print(f"95th percentile: {np.percentile(headline_lengths, 95):.0f}")
print(f"Max: {max(headline_lengths)}")

Mean: 22.4
Median: 22.0
95th percentile: 31
Max: 54


## 4. Hyperparameters

In [54]:
# Model hyperparameters
MAX_TEXT_LEN = 256
MAX_HEAD_LEN = 25
HIDDEN_DIM = 512
NUM_LAYERS = 1
BATCH_SIZE = 32
EPOCHS = 8
LEARNING_RATE = 0.0005
TEACHER_FORCING_RATIO = 0.4
GRADIENT_CLIP = 1.0

print("Hyperparameters:")
print(f"  Hidden dim: {HIDDEN_DIM}")
print(f"  Num layers: {NUM_LAYERS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Teacher forcing ratio: {TEACHER_FORCING_RATIO}")

Hyperparameters:
  Hidden dim: 512
  Num layers: 1
  Batch size: 32
  Teacher forcing ratio: 0.4


## 5. Dataset and Data Loading

In [33]:
def encode_sentence(tokens, max_len, add_sos_eos=False):
    """Convert tokens to indices"""
    ids = [word2idx.get(t, word2idx["<unk>"]) for t in tokens]

    if add_sos_eos:
        ids = [word2idx["<sos>"]] + ids + [word2idx["<eos>"]]

    # Pad or truncate
    if len(ids) < max_len:
        ids += [word2idx["<pad>"]] * (max_len - len(ids))
    else:
        ids = ids[:max_len]

    return ids

class HeadlineDataset(Dataset):
    def __init__(self, texts, headlines):
        self.texts = texts
        self.headlines = headlines

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # Encode source text
        src = torch.tensor(encode_sentence(self.texts[idx], MAX_TEXT_LEN), dtype=torch.long)
        # Encode target headline with <sos> and <eos>
        trg = torch.tensor(encode_sentence(self.headlines[idx], MAX_HEAD_LEN, add_sos_eos=True), dtype=torch.long)

        # Prepare decoder input and target
        decoder_input = trg[:-1]   # <sos> ... second-to-last token
        decoder_target = trg[1:]   # first token ... <eos>
        return src, decoder_input, decoder_target

In [34]:
# Split into train/val
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_headlines, val_headlines = train_test_split(
    tokenized_texts, tokenized_headlines, test_size=0.1, random_state=42
)

train_dataset = HeadlineDataset(train_texts, train_headlines)
val_dataset = HeadlineDataset(val_texts, val_headlines)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train size: {len(train_dataset)}")
print(f"Val size: {len(val_dataset)}")

Train size: 15240
Val size: 1694


## 6. Model with Attention (FIXED)

In [35]:
class Seq2SeqAttnLSTM(nn.Module):
    """FIXED: Improved attention mechanism"""
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers,
                 embedding_matrix=None, freeze_embeddings=False):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # Embedding
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=word2idx["<pad>"])
        if embedding_matrix is not None:
            self.embedding.weight.data.copy_(embedding_matrix)
        self.embedding.weight.requires_grad = not freeze_embeddings

        # Encoder (bidirectional for better context)
        self.encoder = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers,
                               batch_first=True, bidirectional=True, dropout=0.3 if num_layers > 1 else 0)

        # Decoder
        self.decoder = nn.LSTM(embedding_dim + hidden_dim * 2, hidden_dim,
                               num_layers=num_layers, batch_first=True,
                               dropout=0.3 if num_layers > 1 else 0)

        # Attention
        self.attention = nn.Linear(hidden_dim * 3, 1)  # decoder_hidden + encoder_output

        # Output projection
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(0.3)

        # Bridge layer to convert bidirectional encoder states to decoder states
        self.bridge_h = nn.Linear(hidden_dim * 2, hidden_dim)
        self.bridge_c = nn.Linear(hidden_dim * 2, hidden_dim)

    def forward(self, src, trg_input, teacher_forcing_ratio=0.5):
        """
        src: (batch, src_len)
        trg_input: (batch, trg_len)
        teacher_forcing_ratio: probability of using teacher forcing
        """
        batch_size = src.size(0)
        trg_len = trg_input.size(1)

        # Encoder
        embedded_src = self.dropout(self.embedding(src))  # (batch, src_len, emb_dim)
        enc_outputs, (hidden, cell) = self.encoder(embedded_src)
        # enc_outputs: (batch, src_len, hidden_dim * 2) because bidirectional

        # Convert bidirectional encoder states to decoder states
        # hidden/cell: (num_layers * 2, batch, hidden_dim)
        hidden = hidden.view(self.num_layers, 2, batch_size, self.hidden_dim)
        cell = cell.view(self.num_layers, 2, batch_size, self.hidden_dim)

        # Concatenate forward and backward
        hidden = torch.cat([hidden[:, 0, :, :], hidden[:, 1, :, :]], dim=2)  # (num_layers, batch, hidden_dim*2)
        cell = torch.cat([cell[:, 0, :, :], cell[:, 1, :, :]], dim=2)

        # Project to decoder hidden size
        hidden = torch.tanh(self.bridge_h(hidden))  # (num_layers, batch, hidden_dim)
        cell = torch.tanh(self.bridge_c(cell))

        # Decoder with attention
        embedded_trg = self.dropout(self.embedding(trg_input))  # (batch, trg_len, emb_dim)
        outputs = torch.zeros(batch_size, trg_len, vocab_size).to(src.device)

        # Initial input is <sos> token
        dec_input = embedded_trg[:, 0, :].unsqueeze(1)  # (batch, 1, emb_dim)

        for t in range(trg_len):
            # Attention mechanism
            # Repeat decoder hidden state for each encoder timestep
            hidden_repeated = hidden[-1].unsqueeze(1).repeat(1, enc_outputs.size(1), 1)  # (batch, src_len, hidden_dim)

            # Concatenate with encoder outputs
            attn_input = torch.cat([hidden_repeated, enc_outputs], dim=2)  # (batch, src_len, hidden_dim*3)
            attn_weights = self.attention(attn_input).squeeze(2)  # (batch, src_len)
            attn_weights = F.softmax(attn_weights, dim=1)  # (batch, src_len)

            # Compute context vector
            context = torch.bmm(attn_weights.unsqueeze(1), enc_outputs)  # (batch, 1, hidden_dim*2)

            # Concatenate context with input embedding
            rnn_input = torch.cat([dec_input, context], dim=2)  # (batch, 1, emb_dim + hidden_dim*2)

            # Decoder step
            output, (hidden, cell) = self.decoder(rnn_input, (hidden, cell))

            # Project to vocabulary
            prediction = self.fc(self.dropout(output.squeeze(1)))  # (batch, vocab_size)
            outputs[:, t, :] = prediction

            # Teacher forcing: use actual target as next input
            use_teacher_forcing = torch.rand(1).item() < teacher_forcing_ratio
            if use_teacher_forcing and t < trg_len - 1:
                dec_input = embedded_trg[:, t + 1, :].unsqueeze(1)
            else:
                # Use model's own prediction
                top1 = prediction.argmax(1)
                dec_input = self.embedding(top1).unsqueeze(1)

        return outputs

## 7. Training with Validation (NEW)

In [36]:
# Initialize model
model = Seq2SeqAttnLSTM(
    vocab_size, EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS,
    embedding_matrix=embedding_matrix, freeze_embeddings=True
)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=word2idx["<pad>"])
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

print(f"Model has {sum(p.numel() for p in model.parameters())} parameters")
print(f"Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad)} parameters")

Model has 7561677 parameters
Trainable: 4836477 parameters


In [55]:
def train_epoch(model, loader, criterion, optimizer, device, teacher_forcing_ratio):
    """Train for one epoch"""
    model.train()
    total_loss = 0

    for src, dec_input, dec_target in tqdm(loader, desc="Training"):
        src = src.to(device)
        dec_input = dec_input.to(device)
        dec_target = dec_target.to(device)

        optimizer.zero_grad()

        # Forward pass with teacher forcing
        output = model(src, dec_input, teacher_forcing_ratio=teacher_forcing_ratio)

        # Compute loss
        loss = criterion(output.reshape(-1, vocab_size), dec_target.reshape(-1))

        # Backward pass
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)

        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)

def validate(model, loader, criterion, device):
    """Validate the model"""
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for src, dec_input, dec_target in tqdm(loader, desc="Validating"):
            src = src.to(device)
            dec_input = dec_input.to(device)
            dec_target = dec_target.to(device)

            # No teacher forcing during validation
            output = model(src, dec_input, teacher_forcing_ratio=0.0)
            loss = criterion(output.reshape(-1, vocab_size), dec_target.reshape(-1))
            total_loss += loss.item()

    return total_loss / len(loader)

In [56]:
# Training loop
best_val_loss = float('inf')
train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"{'='*60}")

    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, DEVICE, TEACHER_FORCING_RATIO)
    train_losses.append(train_loss)

    # Validate
    val_loss = validate(model, val_loader, criterion, DEVICE)
    val_losses.append(val_loss)

    # Update learning rate
    scheduler.step(val_loss)

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
        }, 'best_model.pth')
        print("✓ Saved best model")


Epoch 1/8


Validating: 100%|██████████| 53/53 [00:01<00:00, 29.63it/s]


Train Loss: 4.4435
Val Loss: 5.2402
Learning Rate: 0.000031
✓ Saved best model

Epoch 2/8


Validating: 100%|██████████| 53/53 [00:01<00:00, 29.50it/s]


Train Loss: 4.4352
Val Loss: 5.2474
Learning Rate: 0.000031

Epoch 3/8


...

Train Loss: 4.4291
Val Loss: 5.2255
Learning Rate: 0.000031
✓ Saved best model

Epoch 7/8


Validating: 100%|██████████| 53/53 [00:01<00:00, 30.14it/s]


Train Loss: 4.4250
Val Loss: 5.2238
Learning Rate: 0.000031
✓ Saved best model

Epoch 8/8


Validating: 100%|██████████| 53/53 [00:02<00:00, 24.53it/s]


Train Loss: 4.4146
Val Loss: 5.2168
Learning Rate: 0.000031
✓ Saved best model


## 8. Inference with Beam Search (NEW)

In [57]:
def beam_search_decode(model, src, beam_width=5, max_len=MAX_HEAD_LEN):
    """
    Beam search decoding for better headline generation
    """
    model.eval()

    with torch.no_grad():
        # Encode
        embedded_src = model.embedding(src)
        enc_outputs, (hidden, cell) = model.encoder(embedded_src)

        # Convert bidirectional to decoder states
        batch_size = src.size(0)
        hidden = hidden.view(model.num_layers, 2, batch_size, model.hidden_dim)
        cell = cell.view(model.num_layers, 2, batch_size, model.hidden_dim)
        hidden = torch.cat([hidden[:, 0, :, :], hidden[:, 1, :, :]], dim=2)
        cell = torch.cat([cell[:, 0, :, :], cell[:, 1, :, :]], dim=2)
        hidden = torch.tanh(model.bridge_h(hidden))
        cell = torch.tanh(model.bridge_c(cell))

        # Initialize beam
        beams = [(torch.tensor([[word2idx["<sos>"]]], device=src.device),
                  0.0, hidden, cell)]  # (sequence, score, hidden, cell)

        completed_beams = []

        for _ in range(max_len):
            all_candidates = []

            for seq, score, h, c in beams:
                if seq[0, -1].item() == word2idx["<eos>"]:
                    completed_beams.append((seq, score))
                    continue

                # Get last token
                last_token = seq[:, -1:]
                dec_input = model.embedding(last_token)

                # Attention
                hidden_repeated = h[-1].unsqueeze(1).repeat(1, enc_outputs.size(1), 1)
                attn_input = torch.cat([hidden_repeated, enc_outputs], dim=2)
                attn_weights = model.attention(attn_input).squeeze(2)
                attn_weights = F.softmax(attn_weights, dim=1)
                context = torch.bmm(attn_weights.unsqueeze(1), enc_outputs)
                rnn_input = torch.cat([dec_input, context], dim=2)

                # Decoder step
                output, (h_new, c_new) = model.decoder(rnn_input, (h, c))
                logits = model.fc(output.squeeze(1))
                log_probs = F.log_softmax(logits, dim=-1)

                # Get top k candidates
                topk_log_probs, topk_ids = log_probs.topk(beam_width, dim=-1)

                for i in range(beam_width):
                    token_id = topk_ids[0, i].unsqueeze(0).unsqueeze(0)
                    token_score = topk_log_probs[0, i].item()
                    new_seq = torch.cat([seq, token_id], dim=1)
                    new_score = score + token_score
                    all_candidates.append((new_seq, new_score, h_new, c_new))

            # Select top beam_width candidates
            beams = sorted(all_candidates, key=lambda x: x[1], reverse=True)[:beam_width]

            # Stop if all beams completed
            if len(beams) == 0:
                break

        # Add remaining beams to completed
        completed_beams.extend([(seq, score) for seq, score, _, _ in beams])

        # Return best sequence
        if completed_beams:
            best_seq = max(completed_beams, key=lambda x: x[1] / len(x[0]))[0]
            return best_seq[0].tolist()
        else:
            return beams[0][0][0].tolist()

def generate_headline(model, raw_text, beam_width=5, max_len=MAX_HEAD_LEN):
    """
    Generate Myanmar headline using beam search
    """
    model.eval()

    # Preprocess
    tokens = processor.tokenize(raw_text, use_dict_merge=False, remove_stopwords=False)
    src_ids = torch.tensor(encode_sentence(tokens, MAX_TEXT_LEN), dtype=torch.long).unsqueeze(0).to(DEVICE)

    # Generate with beam search
    generated_ids = beam_search_decode(model, src_ids, beam_width=beam_width, max_len=max_len)

    # Convert to tokens
    generated_tokens = []
    for idx in generated_ids:
        if idx == word2idx["<eos>"]:
            break
        if idx not in [word2idx["<sos>"], word2idx["<unk>"], word2idx["<pad>"]]:
            generated_tokens.append(idx2word[idx])

    # Join tokens
    return ''.join(generated_tokens)

## 9. Evaluation with BLEU Score (NEW)

In [58]:
def evaluate_bleu(model, val_texts, val_headlines, num_samples=100):
    """
    Evaluate model using BLEU score
    """
    bleu = BLEU()
    generated = []
    references = []

    model.eval()
    for i in tqdm(range(min(num_samples, len(val_texts))), desc="Evaluating BLEU"):
        # Generate headline
        text_tokens = val_texts[i]
        raw_text = ' '.join(text_tokens)
        pred = generate_headline(model, raw_text, beam_width=3)

        # Get reference
        ref = ''.join(val_headlines[i])

        generated.append(pred)
        references.append([ref])  # BLEU expects list of references

        if i < 5:  # Print first 5 examples
            print(f"\n--- Example {i+1} ---")
            print(f"Reference: {ref}")
            print(f"Generated: {pred}")

    # Compute BLEU
    score = bleu.corpus_score(generated, references)
    print(f"\n{'='*60}")
    print(f"BLEU Score: {score.score:.2f}")
    print(f"{'='*60}")

    return score.score

In [59]:
# Evaluate on validation set
bleu_score = evaluate_bleu(model, val_texts, val_headlines, num_samples=100)

Evaluating BLEU:   4%|▍         | 4/100 [00:00<00:05, 18.17it/s]


--- Example 1 ---
Reference: လက်နက်မှုဖြင့်စွပ်စွဲဖမ်းထားသူ၁ဦးကိုထောင်ထဲမှခေါ်ထုတ်ပြီးပစ်သတ်
Generated: ဖမ်းဆီးခံရရသည့်ကိုရဲတပ်ကရဲတပ်ကဖမ်းဆီး

--- Example 2 ---
Reference: မြန်မာ့ပြောင်းစျေးကောင်းမရ၊မူဝါဒအသစ်ကြောင့်ပြောင်းကုန်သည်များအရှုံးပေါ်နိုင်
Generated: မြန်မာ့ဆန်ဈေးနှုန်းကျဆင်း

--- Example 3 ---
Reference: ပေါ်တူဂီတွင်လစာတိုးမြှင့်ရရှိရန်ဆရာ/မများချီတက်ဆန္ဒပြ
Generated: ဘရာဇီးတွင်ပညာရေးဝန်ထမ်းများများဆန္ဒပြ

--- Example 4 ---
Reference: ငလျင်ကြောင့်မင်းကွန်းရှိရှေးဟောင်းစေတီပုထိုးများပျက်စီး
Generated: မန္တလေးမြို့အနီးမြေငလျင်လှုပ်


Evaluating BLEU:   8%|▊         | 8/100 [00:00<00:05, 17.01it/s]


--- Example 5 ---
Reference: ရတနာပုံတက္ကသိုလ်ပါမောက္ခချုပ်အိမ်ကိုရဲနှင့်စစ်တပ်ဝင်စီး
Generated: မန္တလေးတွင်က္ကသိုလ်ကျောင်းက္ကသိုလ်ကျောင်းက္ကသိုလ်ကျောင်းက္ကသိုလ်ကျောင်းသား၁ဦးသေဆုံး


Evaluating BLEU: 100%|██████████| 100/100 [00:05<00:00, 17.00it/s]


BLEU Score: 0.00


## 10. Test on Example

In [67]:
# Test on your example
test_text = "ရုရှားရဲ့ ထင်ရှားကျော်ကြားတဲ့ အတိုက်အခံ နိုင်ငံရေးသမား အသက် ၄၇ နှစ်အရွယ် အလက်ဇီ နာဗယ်လ်ညီရဲ့ သေဆုံးမှုအတွက် လျှို့ဝှက်ဈာပန လုပ်ဖို့ ရုရှားစုံစမ်းစစ်ဆေးရေးမှူးတွေက စီစဉ် နေကြောင်း နာဗယ်လ်ညီရဲ့ မိခင် လျုဒ်မီလာက ဖေဖော်ဝါရီ ၂၂ ရက်မှာ စွပ်စွဲပြောဆိုလိုက်ပြီး ဒါကို သဘောတူမှာ မဟုတ်ဘူးလို့လည်း ပြောကြားခဲ့ပါတယ်။ “သူတို့က လျှို့လျှို့ဝှက်ဝှက် တိတ်တိတ်ဆိတ်ဆိတ် ဖြစ်စေချင်ပြီးတော့ အောက်မေ့ဖွယ် အခမ်းအနားလည်း မလုပ်စေချင်ဘူး။ သူတို့က သုသာန်အစွန်ပိုင်းက လတ်လတ်ဆတ်ဆတ် သင်္ချိုင်းတခုကို ခေါ်သွားပြီးတော့ ဒီမှာ ခင်ဗျားရဲ့သား လဲလျောင်းနေတယ် ဆိုတာမျိုး ပြောချင်တာ၊ ဒါကို ငါသဘောမတူဘူး”လို့ ယူကျုဘ် ဗီဒီယိုတခုထဲမှာ လျုဒ်မီလာက ပြောပါတယ်။ ကျမမျက်လုံးတွေကို ကြည့်ပြီးတော့ စုံစမ်းစစ်ဆေးရေးမှူးက လျှို့ဝှက်အသုဘအခမ်းအနားလုပ်ဖို့ သဘောမတူရင် ကျမသားရဲ့အလောင်းကို သူတို့ တစ်ခုခုလုပ်မယ်လို့ ပြောကြောင်းလည်း နာဗာလ်နီရဲ့ မိခင် လျုဒ်မီလာက ပြောပါတယ်။ “ဥပဒေအရတော့ အလက်ဇီရဲ့ အလောင်းကို ချက်ချင်း ပေးသင့်တယ်၊ ဒါပေမယ့် မပေးကြဘူး။ အဲဒီအစား ခြိမ်းခြောက်ပြောဆိုပြီးတော့ အလက်ဇီကို ဘယ်နေရာ၊ ဘယ်အချိန်မှာ ဘယ်လို သင်္ဂြိုဟ်ရမလဲဆိုတဲ့ အခြေအနေတွေ ပေးတယ်။ ဒါက တရားမဝင်ဘူး။ သူတို့က ကရင်မလင်က ဒါမှမဟုတ် စုံစမ်းစစ်ဆေးရေးကော်မတီ ဗဟိုရုံးမှ အမိန့်တွေကို လက်ခံရရှိထားလို့ဖြစ်မယ်”ဆိုပြီး လျုဒ်မီလာက ပြောဆိုပါတယ်။ စစ်ဆေးရေးမှူး ဗိုရိုပါယဗ်ကတော့ အချိန်မရတော့ဘူး၊ အလောင်းပျက်စီးသွားပြီလို့ ပွင့်ပွင့်လင်းလင်း ပြောခဲ့တယ်လို့လည်း လျုဒ်မီလာက ပြောပါတယ်။ သေဆုံးသူ နာဗယ်လ်ညီရဲ့ မိခင် လျုဒ်မီလာ နာဗယ်လ် နာယာကတော့ သူ့သားအလောင်းကို နောက်ဆုံး တွေ့မြင်ခွင့်ရအောင် လွှဲပေးဖို့ ရုရှားသမ္မတ ဗလာဒီမီယာ ပူတင်ကို ဗီဒီယို မက်ဆေ့ချ်ထဲမှာ တောင်းဆိုခဲ့ပါတယ်။ ရုရှားစုံစမ်းစစ်ဆေးရေးမှူးတွေဆီက ချက်ချင်းတုံ့ပြန်မှု မရှိခဲ့ပါဘူ။ နာဗယ်လ်ညီရဲ့ အလောင်းနဲ့ သေဆုံးကြောင်း လက်မှတ်ကို မိခင်ဖြစ်သူ လျုဒ်မီလာကို ပြသခဲ့ပြီး နာဗာလ်နီဟာ သဘာဝ အကြောင်းတရားကြောင့် သေဆုံးခဲ့ရကြောင်း သေစာရင်းထဲမှာ ဖော်ပြထားတယ်လို့ နာဗယ်လ်ညီ ရဲ့အဖွဲ့မှ ပြောပါတယ်။ နာဗယ်လ်ညီရဲ့ သေဆုံးမှုဟာ ကရင်မလင်က သူ့ကို သတ်ခဲ့တယ်လို့ သူ့အဖွဲ့နဲ့ မိသားစုက စွပ်စွဲထားပြီး ကရင်မလင်က ဒေါသတကြီး ပယ်ချခဲ့ပါတယ်။ Source/Photo - Reuters "
print("Article:")
print(test_text)
print("\n" + "="*60)
print("Generated Headline:")
headline = generate_headline(model, test_text, beam_width=10)
print(headline)
print("="*60)

Article:
ရုရှားရဲ့ ထင်ရှားကျော်ကြားတဲ့ အတိုက်အခံ နိုင်ငံရေးသမား အသက် ၄၇ နှစ်အရွယ် အလက်ဇီ နာဗယ်လ်ညီရဲ့ သေဆုံးမှုအတွက် လျှို့ဝှက်ဈာပန လုပ်ဖို့ ရုရှားစုံစမ်းစစ်ဆေးရေးမှူးတွေက စီစဉ် နေကြောင်း နာဗယ်လ်ညီရဲ့ မိခင် လျုဒ်မီလာက ဖေဖော်ဝါရီ ၂၂ ရက်မှာ စွပ်စွဲပြောဆိုလိုက်ပြီး ဒါကို သဘောတူမှာ မဟုတ်ဘူးလို့လည်း ပြောကြားခဲ့ပါတယ်။ “သူတို့က လျှို့လျှို့ဝှက်ဝှက် တိတ်တိတ်ဆိတ်ဆိတ် ဖြစ်စေချင်ပြီးတော့ အောက်မေ့ဖွယ် အခမ်းအနားလည်း မလုပ်စေချင်ဘူး။ သူတို့က သုသာန်အစွန်ပိုင်းက လတ်လတ်ဆတ်ဆတ် သင်္ချိုင်းတခုကို ခေါ်သွားပြီးတော့ ဒီမှာ ခင်ဗျားရဲ့သား လဲလျောင်းနေတယ် ဆိုတာမျိုး ပြောချင်တာ၊ ဒါကို ငါသဘောမတူဘူး”လို့ ယူကျုဘ် ဗီဒီယိုတခုထဲမှာ လျုဒ်မီလာက ပြောပါတယ်။ ကျမမျက်လုံးတွေကို ကြည့်ပြီးတော့ စုံစမ်းစစ်ဆေးရေးမှူးက လျှို့ဝှက်အသုဘအခမ်းအနားလုပ်ဖို့ သဘောမတူရင် ကျမသားရဲ့အလောင်းကို သူတို့ တစ်ခုခုလုပ်မယ်လို့ ပြောကြောင်းလည်း နာဗာလ်နီရဲ့ မိခင် လျုဒ်မီလာက ပြောပါတယ်။ “ဥပဒေအရတော့ အလက်ဇီရဲ့ အလောင်းကို ချက်ချင်း ပေးသင့်တယ်၊ ဒါပေမယ့် မပေးကြဘူး။ အဲဒီအစား ခြိမ်းခြောက်ပြောဆိုပြီးတော့ အလက်ဇီကို ဘယ်နေရာ၊ ဘယ်အချိန်မှာ ဘယ်လို သင်္ဂြိုဟ်ရမလဲဆိုတဲ့ အခြ

## 11. Save Model and Artifacts

In [62]:
# Save model
torch.save(model.state_dict(), "seq2seq_model_improved.pth")

# Save vocabulary
with open("vocab_improved.pkl", "wb") as f:
    pickle.dump({
        "word2idx": word2idx,
        "idx2word": idx2word
    }, f)

# Save processor info
processor_info = {
    "dict_path": DICT_PATH,
    "stopwords_path": STOPWORDS_PATH
}
with open("processor_improved.pkl", "wb") as f:
    pickle.dump(processor_info, f)

# Save training history
with open("training_history.pkl", "wb") as f:
    pickle.dump({
        "train_losses": train_losses,
        "val_losses": val_losses,
        "bleu_score": bleu_score
    }, f)

print("✓ All artifacts saved!")

✓ All artifacts saved!


In [63]:
import torch
import pickle
from google.colab import drive
import os

# --- Mount Google Drive ---
drive.mount('/content/drive')

# --- Choose a folder in your Drive ---
drive_folder = "/content/drive/MyDrive/seq2seq_headline_model"
os.makedirs(drive_folder, exist_ok=True)

# --- Save model ---
torch.save(model.state_dict(), os.path.join(drive_folder, "seq2seq_model_improved.pth"))

# --- Save vocabulary ---
with open(os.path.join(drive_folder, "vocab_improved.pkl"), "wb") as f:
    pickle.dump({
        "word2idx": word2idx,
        "idx2word": idx2word
    }, f)

# --- Save processor info ---
processor_info = {
    "dict_path": DICT_PATH,
    "stopwords_path": STOPWORDS_PATH
}
with open(os.path.join(drive_folder, "processor_improved.pkl"), "wb") as f:
    pickle.dump(processor_info, f)

# --- Save training history ---
with open(os.path.join(drive_folder, "training_history.pkl"), "wb") as f:
    pickle.dump({
        "train_losses": train_losses,
        "val_losses": val_losses,
        "bleu_score": bleu_score
    }, f)

print(f"✓ All artifacts saved to {drive_folder} ✅")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ All artifacts saved to /content/drive/MyDrive/seq2seq_headline_model ✅


## Summary of Improvements

### 1. **Fixed Preprocessing Consistency**
   - Used same regex pattern for both text and headlines
   - Consistent tokenization strategy

### 2. **Added Teacher Forcing**
   - 50% teacher forcing ratio during training
   - Helps model learn better dependencies

### 3. **Improved Attention**
   - Bidirectional encoder for better context
   - Fixed attention computation
   - Proper state bridging from encoder to decoder

### 4. **Added Validation**
   - Train/val split (90/10)
   - Monitor overfitting
   - Save best model

### 5. **Beam Search Inference**
   - Better than greedy decoding
   - Explores multiple hypotheses

### 6. **BLEU Evaluation**
   - Quantitative metric
   - Track generation quality

### 7. **Better Training**
   - Learning rate scheduling
   - Gradient clipping
   - Dropout for regularization

This should significantly improve headline quality!